# Λ Proyecto Lambda | Optimización de Inversiones Petroleras

## Contexto empresarial

OilyGiant es una compañía de exploración y extracción de petróleo que busca expandir sus operaciones mediante el desarrollo de nuevos pozos petroleros. Debido al elevado costo de inversión asociado a la perforación, resulta fundamental seleccionar cuidadosamente las regiones con mayor potencial económico y menor riesgo financiero.

La compañía dispone de datos geológicos obtenidos de tres regiones diferentes. A partir de esta información se desarrollarán modelos predictivos capaces de estimar las reservas de petróleo de nuevos pozos, permitiendo identificar las ubicaciones con mayor potencial de rentabilidad.

Además del volumen esperado de reservas, la empresa necesita evaluar el riesgo asociado a cada región para garantizar que la inversión propuesta mantenga una probabilidad de pérdida aceptablemente baja.

## Objetivos

- Analizar y preparar los datos geológicos disponibles.
- Construir modelos de regresión para estimar reservas de petróleo.
- Evaluar el desempeño predictivo mediante RMSE.
- Identificar los 200 pozos más prometedores dentro de cada región.
- Estimar los beneficios potenciales de cada región.
- Aplicar técnicas de simulación mediante bootstrapping.
- Calcular beneficios promedio, intervalos de confianza y riesgo de pérdidas.
- Identificar la región con mejor equilibrio entre rentabilidad y riesgo.
- Formular una recomendación de inversión basada en evidencia cuantitativa.

## Conjuntos de datos utilizados

- `geo_data_0.csv`
- `geo_data_1.csv`
- `geo_data_2.csv`

## Variables disponibles

- `id` — identificador único del pozo.
- `f0` — característica geológica.
- `f1` — característica geológica.
- `f2` — característica geológica.
- `product` — volumen de reservas de petróleo (miles de barriles).

## Tipo de problema

**Regresión supervisada**

El objetivo consiste en predecir:

```text
Volumen de reservas de petróleo
```

para nuevos pozos petroleros.

## Modelo utilizado

Según las condiciones del proyecto:

```text
Linear Regression
```

## Métrica principal

```text
RMSE
```

## Pregunta de negocio

¿Cuál de las tres regiones ofrece la mejor combinación de rentabilidad esperada y bajo riesgo para desarrollar 200 nuevos pozos petroleros?

Primero traemos nuestros datasets de trabajo

In [ ]:
import pandas as pd

geo_data_0 = pd.read_csv("/content/geo_data_0.csv")
geo_data_1 = pd.read_csv("/content/geo_data_1.csv")
geo_data_2 = pd.read_csv("/content/geo_data_2.csv")

Procedemos a una exploración inicial de su contenido

In [ ]:
print(geo_data_0.info())
print()
print()
print(geo_data_1.info())
print()
print()
print(geo_data_2.info())
print()
print()
print(geo_data_0.sample(8))
print()
print()
print(geo_data_1.sample(8))
print()
print()
print(geo_data_2.sample(8))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB
None


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB
None


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Colu

In [ ]:
geo_data_0['product'].nunique()
geo_data_1['product'].nunique()
geo_data_2['product'].nunique()

100000

Después de un análisis general de los tres datasets, se observó que no existen variables categóricas que requieran la implementación del One Hot Encoding. Se podria decir que la única característica que podría considerarse categórica no es binaria sino única (el id), y como tal no representa un valor que deba tomarse muy en cuenta a la hora de generar el modelo predictor.

Se identificó la variable objetivo como "product" y al ser un valor continuo nos decantamos desde ya por el modelo de regresión logística.

No se identificaron valores faltantes o ausentes que debamos tratar.

Por último se observó que en la variable predictora no existen tantos valores continuos para un dataset de 100.000 observaciones, ya que solamente tenemos 12 valores únicos. Se considera que no es necesaria la manipulación de estos datos pero si es algo que tendremos en cuenta.

Antes de construir el modelo se tomó en cuenta que la única variable que no aporta valor real al modelo a construir es la columna con el identificador único "id" por lo que se eliminará de los tres datasets.

In [ ]:
dataset_limpio_0 = geo_data_0.drop(["id"], axis = 1).copy()
dataset_limpio_1 = geo_data_1.drop(["id"], axis = 1).copy()
dataset_limpio_2 = geo_data_2.drop(["id"], axis = 1).copy()

print(dataset_limpio_0.head(5))
print()
print()
print(dataset_limpio_1.head(5))
print()
print()
print(dataset_limpio_2.head(5))

         f0        f1        f2     product
0  0.705745 -0.497823  1.221170  105.280062
1  1.334711 -0.340164  4.365080   73.037750
2  1.022732  0.151990  1.419926   85.265647
3 -0.032172  0.139033  2.978566  168.620776
4  1.988431  0.155413  4.751769  154.036647


          f0         f1        f2     product
0 -15.001348  -8.276000 -0.005876    3.179103
1  14.272088  -3.475083  0.999183   26.953261
2   6.263187  -5.948386  5.001160  134.766305
3 -13.081196 -11.506057  4.999415  137.945408
4  12.702195  -8.147433  5.004363  134.766305


         f0        f1        f2     product
0 -1.146987  0.963328 -0.828965   27.758673
1  0.262778  0.269839 -2.530187   56.069697
2  0.194587  0.289035 -5.586433   62.871910
3  2.236060 -0.553760  0.930038  114.572842
4 -0.515993  1.716266  5.899011  149.600746


Ahora procederemos con la definición de la variable objetivos y las variables predictoras antes de separar los datos en los conjuntos de entrenamiento, validación y prueba



In [ ]:
objetivo_0 = dataset_limpio_0["product"]
caracteristicas_0 = dataset_limpio_0.drop(["product"], axis = 1)

objetivo_1 = dataset_limpio_1["product"]
caracteristicas_1 = dataset_limpio_1.drop(["product"], axis = 1)

objetivo_2 = dataset_limpio_2["product"]
caracteristicas_2 = dataset_limpio_2.drop(["product"], axis = 1)

Siguiendo las instrucciones del proyecto separaremos el dataset en un conjunto de entrenamiento y uno de validación con una proporción de 75:25.



In [ ]:
from sklearn.model_selection import train_test_split

caracteristicas_0_entrenamiento, caracteristicas_0_validacion, objetivo_0_entrenamiento, objetivo_0_validacion = train_test_split(caracteristicas_0, objetivo_0, train_size = 0.75, test_size = 0.25, random_state = 12345, shuffle = True)


En este momento nos enfocaremos en la construcción del modelo elegido con su posterior entrenamiento y validación.

In [ ]:
from sklearn.linear_model import LinearRegression

model_lineal = LinearRegression()

model_lineal.fit(caracteristicas_0_entrenamiento, objetivo_0_entrenamiento)

prediccion_valida_0 = model_lineal.predict(caracteristicas_0_validacion)


Tenemos 25 mil predicciones para el conjunto de validación. Sin embargo nos enfocaremos en encontrar el volumen medio de reservas en el pozo de petroleo

In [ ]:
import numpy as np

volumen_medio_geo_0 = np.mean(prediccion_valida_0)

print("El volumen medio de reservas en el pozo de petroleo es de:", volumen_medio_geo_0)

El volumen medio de reservas en el pozo de petroleo es de: 92.59256778438035


Hasta ahora el modelo estima que la region 0 generará en promedio 92.6 mil barriles de reserva para pozos nuevos.

No nos dentendremos aquí, dado que nos falta calcular el error cuadrático medio

In [ ]:
from sklearn.metrics import mean_squared_error

ECM_0 = mean_squared_error(objetivo_0_validacion, prediccion_valida_0)

RECM = ECM_0 ** 0.5

print("La raíz del error cuadrático medio de esta predicción lineal es de:", RECM)

La raíz del error cuadrático medio de esta predicción lineal es de: 37.5794217150813


podemos ver ahora que en promedio, las predicciones del modelo se desvían aproximadamente 37.58 mil barriles respecto al valor real.

Ahora obtendremos estos mismos resultados para el dataset_1 y el dataset_2 mediante funciones

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np


def entrenar_evaluar_region(caracteristicas, objetivo):

    # División entrenamiento-validación
    caracteristicas_entrenamiento, caracteristicas_validacion, objetivo_entrenamiento, objetivo_validacion = train_test_split(
        caracteristicas,
        objetivo,
        train_size=0.75,
        test_size=0.25,
        random_state=12345,
        shuffle=True
    )

    # Entrenamiento del modelo
    modelo = LinearRegression()

    modelo.fit(
        caracteristicas_entrenamiento,
        objetivo_entrenamiento
    )

    # Predicciones
    predicciones = modelo.predict(caracteristicas_validacion)

    # Volumen promedio predicho
    volumen_medio = np.mean(predicciones)

    # RECM
    ecm = mean_squared_error(
        objetivo_validacion,
        predicciones
    )

    recm = ecm ** 0.5

    return (
        predicciones,
        objetivo_validacion,
        volumen_medio,
        recm
    )

In [ ]:
predicciones_1, objetivo_validacion_1, volumen_medio_1, recm_1 = entrenar_evaluar_region(caracteristicas_1, objetivo_1)

print("El volumen medio de producción de reserva para la zona 1 es de:", volumen_medio_1)
print()
print()
print("La raíz del error cuadrático medio de esta predicción lineal es de:", recm_1)


El volumen medio de producción de reserva para la zona 1 es de: 68.728546895446


La raíz del error cuadrático medio de esta predicción lineal es de: 0.893099286775617


Para la region 1 el modelo estima que en promedio el volumen de reserva para pozos nuevos es de 68.7 mil barriles.

Y podemos decir que este es un resultado favorable cuando hablamos del RECM dado que en promedio el modelo tiene un error de 0.89 respecto al valor real.

In [ ]:
predicciones_2, objetivo_validacion_2, volumen_medio_2, recm_2 = entrenar_evaluar_region(caracteristicas_2, objetivo_2)
print("El volumen medio de producción de reserva para la zona 2 es de:", volumen_medio_2)
print()
print()
print("La raíz del error cuadrático medio de esta predicción lineal es de:", recm_2)

El volumen medio de producción de reserva para la zona 2 es de: 94.96504596800489


La raíz del error cuadrático medio de esta predicción lineal es de: 40.02970873393434


Para este caso el la estimación promedio del modelo nos dice que la region 2 producirá 96.9 mil barriles de reserva.

Por otro lado el RECM nos señala que el modelo tuvo una desviación de 40 mil barriles del valor real

El modelo de regresión lineal presenta comportamientos distintos entre las regiones analizadas. La región 1 obtuvo un RECM de aproximadamente 0.89 mil barriles, significativamente inferior al de las regiones 0 y 2 (37.58 y 40.03 mil barriles respectivamente). Esto sugiere que las características disponibles explican mucho mejor el volumen de reservas en la región 1, permitiendo una predicción considerablemente más precisa.

En este momento trabajaremos con la sección 3 del proyecto en la cual calcularemos las ganancias

In [ ]:
presupuesto = 100000000
numero_de_pozos = 200
ingreso_de_un_barril = 4.5
ingreso_por_unidad = 4500

# presupuesto por pozo
presupuesto_por_pozo = presupuesto / numero_de_pozos

print("El presupuesto para cada pozo individual es de:", presupuesto_por_pozo)

#punto de equilibrio
punto_de_equilibrio = presupuesto_por_pozo / ingreso_por_unidad
print("El punto de equilibrio es de:", punto_de_equilibrio)

El presupuesto para cada pozo individual es de: 500000.0
El punto de equilibrio es de: 111.11111111111111


Con esta operación en cuenta sabemos que cada pozo debe retornar mínimo 500.000 dolares para que el proyecto no pierda dinero. Y que cada pozo debe retornar un valor mínimo de 111.11 mil barriles para recuperar la inversión.

In [ ]:
# diferencias entre las producciones promedio de los pozos y la producción mínima

diferencia_region_0 = punto_de_equilibrio - volumen_medio_geo_0
print("La diferencia entre el promedio producido para la region 0 y el ideal es de:", diferencia_region_0)
print()
print()
diferencia_region_1 = punto_de_equilibrio - volumen_medio_1
print("La diferencia entre el promedio producido para la region 1 y el ideal es de:", diferencia_region_1)
print()
print()
diferencia_region_2 = punto_de_equilibrio - volumen_medio_2
print("La diferencia entre el promedio producido para la region 2 y el ideal es de:", diferencia_region_2)
print()
print()

La diferencia entre el promedio producido para la region 0 y el ideal es de: 18.518543326730764


La diferencia entre el promedio producido para la region 1 y el ideal es de: 42.38256421566511


La diferencia entre el promedio producido para la region 2 y el ideal es de: 16.146065143106227




Podemos decir ahora que en la producción promedio de barriles de las tres regiones veremos una diferencia superior a los 15 mil barriles para llegar a la producción que asegura un retorno de la inversión. Sin embargo no podemos decidir de forma concluyente cualquiera de las tres regiones nos producirá pérdidas, ya que aún nos falta calcular las ganancias potenciales de los 200 principales por región.

Identificación de los 200 mejores pozos de la región 0

In [ ]:
Series_prediccion_valida_0 = pd.Series(prediccion_valida_0, index = objetivo_0_validacion.index )
Series_prediccion_valida_1 = pd.Series(predicciones_1, index = objetivo_validacion_1.index)
Series_prediccion_valida_2 = pd.Series(predicciones_2, index = objetivo_validacion_2.index)

In [ ]:
mejores_pozos0 = Series_prediccion_valida_0.sort_values(ascending = False)
mejores_pozos1 = Series_prediccion_valida_1.sort_values(ascending = False)
mejores_pozos2 = Series_prediccion_valida_2 .sort_values(ascending = False)


mejores_pozos_0 = mejores_pozos0.head(200)
mejores_pozos_1 = mejores_pozos1.head(200)
mejores_pozos_2 = mejores_pozos2.head(200)


print(mejores_pozos_0.info())
print()
print()
print(mejores_pozos_1.info())
print()
print()
print(mejores_pozos_2.info())
print()
print()
print(mejores_pozos_0.head(10))
print()
print()
print(mejores_pozos_1.head(10))
print()
print()
print(mejores_pozos_2.head(10))

<class 'pandas.core.series.Series'>
Index: 200 entries, 93073 to 75908
Series name: None
Non-Null Count  Dtype  
--------------  -----  
200 non-null    float64
dtypes: float64(1)
memory usage: 3.1 KB
None


<class 'pandas.core.series.Series'>
Index: 200 entries, 38665 to 59892
Series name: None
Non-Null Count  Dtype  
--------------  -----  
200 non-null    float64
dtypes: float64(1)
memory usage: 3.1 KB
None


<class 'pandas.core.series.Series'>
Index: 200 entries, 98619 to 62558
Series name: None
Non-Null Count  Dtype  
--------------  -----  
200 non-null    float64
dtypes: float64(1)
memory usage: 3.1 KB
None


93073    180.180713
46784    176.252213
78948    175.850623
43388    175.658429
6496     173.299686
93716    172.627972
99628    172.501678
11404    169.124234
74946    168.787602
95898    168.206015
dtype: float64


38665    139.818970
20191    139.773423
14041    139.703330
24274    139.560938
92782    139.516754
2242     139.472944
42432    139.472059
86762    139.446501

In [ ]:
volumen_real_0 = objetivo_0_validacion[mejores_pozos_0.index]
volumen_real_1 = objetivo_validacion_1[mejores_pozos_1.index]
volumen_real_2 = objetivo_validacion_2[mejores_pozos_2.index]

print(volumen_real_0.head(5))

93073    162.810993
46784    153.639837
78948    162.153488
43388     96.893581
6496     178.879516
Name: product, dtype: float64


Ahora vamos a calcular la ganancia portencial de los 200 mejores pozos por región

In [ ]:
volumen_total_region_0 = volumen_real_0.sum()
print("Los barriles de reserva para la región 0 son:", volumen_total_region_0)
print()
print()
volumen_total_region_1 = volumen_real_1.sum()
print("Los barriles de reserva para la región 1 son:", volumen_total_region_1)
print()
print()
volumen_total_region_2 = volumen_real_2.sum()
print("Los barriles de reserva para la región 2 son:", volumen_total_region_2)

Los barriles de reserva para la región 0 son: 29601.83565142189


Los barriles de reserva para la región 1 son: 27589.081548181137


Los barriles de reserva para la región 2 son: 28245.22214133296


En este momento podemos ver que la suma de las reservas reales de los 200 pozos seleccionados de la región 0 asciende aproximadamente a 29.6 millones de barriles, para la región 1 unos 27.6 millones de barriles y para la region 2 unos 28.2 millones de barriles

Por lo tanto llegamos a la conslusión de que la región con la mayor cantidad de barriles de reserva es la región 0 por encima de las regiones 1 y 2

Ahora bien, tomando en cuenta que la inversión inicial es de 100 millones de dolares vamos a calcular la ganancia que puede producir cada región con sus 200 mejores pozos

In [ ]:
ganancia_region_0 = (volumen_total_region_0 * 4500) - 100000000
print("Las ganancias para la region 0 son de:", ganancia_region_0)
print()
print()
ganancia_region_1 = (volumen_total_region_1 * 4500) - 100000000
print("Las ganancias para la region 1 son de:", ganancia_region_1)
print()
print()
ganancia_region_2 = (volumen_total_region_2 * 4500) - 100000000
print("Las ganancias para la region 2 son de:", ganancia_region_2)

Las ganancias para la region 0 son de: 33208260.43139851


Las ganancias para la region 1 son de: 24150866.966815114


Las ganancias para la region 2 son de: 27103499.635998324


Habiendo obtenido estos resultados podemos decir que las tres regiones muestran ganancias importantes, siendo la más alta la region 0 con unas ganancias que superan los 30 millones de dolares, ya que las calculamos cerca a 33.2 millones de dolares. Por ahora podemos decir que recomendamos la inversión en la región 0 y el desarrollo de pozos petrolíferos ya que es la única de las tres regiones estudiadas que representará mas de un 30% del retorno del presupuesto del que inicialmente dispone la empresa. Aunque no nos detendremos aquí ya que nos falta calcular el beneficio promedio, el intervalo de confianza del 95% y el riesgo de pérdidas

Uso del Bootstrapping para establecer el intervalo de confianza del 95%

Primero convertiremos objetivo_0_validacion a un Series con el fin de poderlos concatenar y añadirlos como dataset único a las muestras del bootstrapping, ya que en este caso del proyecto son necesarias tanto las predicciones de nuestro modelo como el valor real de los barriles de reserva del pozo en la región

In [ ]:
series_prediccion_valida_0 = pd.Series(prediccion_valida_0, index = objetivo_0_validacion.index)

In [ ]:
muestra_bootstrapping_0 = pd.concat([series_prediccion_valida_0, objetivo_0_validacion], axis = 1)
muestra_bootstrapping_0.columns = ["predicción del modelo", "product"]
print(muestra_bootstrapping_0.info())
print(muestra_bootstrapping_0.head(5))

<class 'pandas.core.frame.DataFrame'>
Index: 25000 entries, 71751 to 43558
Data columns (total 2 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   predicción del modelo  25000 non-null  float64
 1   product                25000 non-null  float64
dtypes: float64(2)
memory usage: 1.1 MB
None
       predicción del modelo     product
71751              95.894952   10.038645
80493              77.572583  114.551489
2655               77.892640  132.603635
53233              90.175134  169.072125
91141              70.510088  122.325180


In [ ]:
import pandas as pd
import numpy as np

muestras_region_0 = muestra_bootstrapping_0
ganancias_boot_region_0 = []
np.random.seed(12345)

for _ in range(1000):
    # Buscamos los 200 mejores pozos
    bootstrapping_200_0 = muestras_region_0.sample(500, replace = True)
    bootstrapping_200_0 = bootstrapping_200_0.sort_values(by = "predicción del modelo", ascending = False)
    bootstrapping_200_0 = bootstrapping_200_0.head(200)

    suma_0 = bootstrapping_200_0["product"].sum()
    multiplicacion_0 = suma_0 * 4500
    resta_0 = multiplicacion_0 - 100000000

    ganancias_boot_region_0.append(resta_0)

Ahora revisaremos que la muestra contenga 1000 registros

In [ ]:
len(ganancias_boot_region_0)

1000

Revisaremos ahora cual es la ganancia promedio de la region 0 con las 1000 muestras en el bootstrapping

In [ ]:
ganancias_promedio_boot_region_0 = np.mean(ganancias_boot_region_0)

print("Las ganancias promedio de la región 0 con el boostrapping son de:", ganancias_promedio_boot_region_0 )

Las ganancias promedio de la región 0 con el boostrapping son de: 3961649.8480237117


Con la implementación de la técnica del Bootstrapping con 1000 simulaciones que incluyen la variabilidad de hechos nos dimos cuenta de un detalle interesante. Encontramos que la ganancia promedio para la región 0 es de 3.96 millones de dolares.

Esto es importante porque no nos limitamos a recomendar la incursión de pozos en esta zona ya que anteriormente tomamos las mejores condiciones, las más favorables. El Bootstrapping nos ofrece un panorama más realista al simular varios escenarios posibles.

Ahora procederemos a calcular el intervalo de confianza del 95%

In [ ]:
inferior = np.percentile(ganancias_boot_region_0, 2.5)
superior = np.percentile(ganancias_boot_region_0, 97.5)

print("IC 95%:")
print(inferior, superior)

IC 95%:
-1112155.4589049604 9097669.41553423


Este resultado nos indica que en el 95% de los casos simulados del Bootstrapping la empresa se encuentra entre pérdidas de 1.11 millones de dólares y ganancias de 9.09 millones de dólares.

Con este resultado nos damos cuenta que aunque en el promedio de los casos la empresa gana 3.96 millones de dólares no todos los escenarios son positivos.

In [ ]:
riesgo_perdidas_0 = (np.array(ganancias_boot_region_0) < 0).mean() * 100

print("El riesgo de pérdidas para la región 0 es del", riesgo_perdidas_0, "%")

El riesgo de pérdidas para la región 0 es del 6.9 %


In [ ]:
probabilidad_ganancia_0 = (np.array(ganancias_boot_region_0) > 0).mean() * 100

print("La probabilidad de ganancias para la región 0 es del", probabilidad_ganancia_0, "%")

La probabilidad de ganancias para la región 0 es del 93.10000000000001 %


En este momento podemos decir que culminamos el análisis de la región 0 y oficialmente determinamos que presenta un riesgo de pérdidas de un 6.9% y una probabilidad de ganancia del 93.10%. Sin embargo al no presentar una probabilidad de riesgo infeririór al 2.5% descartamos esta región

Con los pasos que seguimos para la region 0 en el bootstrapping encontraremos los valores que nos interesan para la siguiente región

In [ ]:
# Conversión de la predicción a series
series_prediccion_valida_1 = pd.Series(predicciones_1, index = objetivo_validacion_1.index)

# Concatenación
muestra_bootstrapping_1 = pd.concat([series_prediccion_valida_1, objetivo_validacion_1], axis = 1)
muestra_bootstrapping_1.columns = ["predicción del modelo", "product"]
print(muestra_bootstrapping_1.info())
print(muestra_bootstrapping_1.head(5))

# Implementación del Bootstrapping
muestras_region_1 = muestra_bootstrapping_1
ganancias_boot_region_1 = []
np.random.seed(12345)

for _ in range(1000):
    # Buscamos los 200 mejores pozos
    bootstrapping_200_1 = muestras_region_1.sample(500, replace = True)
    bootstrapping_200_1 = bootstrapping_200_1.sort_values(by = "predicción del modelo", ascending = False)
    bootstrapping_200_1 = bootstrapping_200_1.head(200)

    suma_1 = bootstrapping_200_1["product"].sum()
    multiplicacion_1 = suma_1 * 4500
    resta_1 = multiplicacion_1 - 100000000

    ganancias_boot_region_1.append(resta_1)
# Verificación de la longitud de la muestra Bootstrapping
len(ganancias_boot_region_1)

# Cálculo de las ganancias promedio de la región
ganancias_promedio_boot_region_1 = np.mean(ganancias_boot_region_1)

print("Las ganancias promedio de la región 1 con el boostrapping son de:", ganancias_promedio_boot_region_1 )

# Determinación del intervalo de confianza
inferior = np.percentile(ganancias_boot_region_1, 2.5)
superior = np.percentile(ganancias_boot_region_1, 97.5)

print("IC 95%:")
print(inferior, superior)


# Cálculo del porcentaje del riesgo de pérdidas
riesgo_perdidas_1 = (np.array(ganancias_boot_region_1) < 0).mean() * 100

print("El riesgo de pérdidas para la región 1 es del", riesgo_perdidas_1, "%")

# Cálculo de porcentaje de la probabilidad de ganancia
probabilidad_ganancia_1 = (np.array(ganancias_boot_region_1) > 0).mean() * 100

print("La probabilidad de ganancias para la región 1 es del", probabilidad_ganancia_1, "%")

<class 'pandas.core.frame.DataFrame'>
Index: 25000 entries, 71751 to 43558
Data columns (total 2 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   predicción del modelo  25000 non-null  float64
 1   product                25000 non-null  float64
dtypes: float64(2)
memory usage: 1.1 MB
None
       predicción del modelo    product
71751              82.663314  80.859783
80493              54.431786  53.906522
2655               29.748760  30.132364
53233              53.552133  53.906522
91141               1.243856   0.000000
Las ganancias promedio de la región 1 con el boostrapping son de: 4560451.057866608
IC 95%:
338205.0939898458 8522894.538660347
El riesgo de pérdidas para la región 1 es del 1.5 %
La probabilidad de ganancias para la región 1 es del 98.5 %


Con esta nueva información podemos ver que la región 1 presenta unas ganancias promedio de 4.56 millones de dólares. Ciertamente un índice de ingresos promedios mayor a la región 0, además podemos observar un índice de confianza más favorable ya que según el boostrapping para 1000 simulaciones en un el 95% de los casos la empresa se encuntra entre ganancias de 0.34 millones de dólares y ganancias de 8.52 millones de dólares.

Para finalizar determinamos que el riesgo de pérdidas se estima en un 1.5 % y la probabilidad de ganancia que resulta bastante alta es del 98.5%.

Por último ejecutamos el análisis final para la region 2

In [ ]:
# Conversión de la predicción a series
series_prediccion_valida_2 = pd.Series(predicciones_2, index = objetivo_validacion_2.index)

# Concatenación
muestra_bootstrapping_2 = pd.concat([series_prediccion_valida_2, objetivo_validacion_2], axis = 1)
muestra_bootstrapping_2.columns = ["predicción del modelo", "product"]
print(muestra_bootstrapping_2.info())
print(muestra_bootstrapping_2.head(5))

# Implementación del Bootstrapping
muestras_region_2 = muestra_bootstrapping_2
ganancias_boot_region_2 = []
np.random.seed(12345)

for _ in range(1000):
    # Buscamos los 200 mejores pozos
    bootstrapping_200_2 = muestras_region_2.sample(500, replace = True)
    bootstrapping_200_2 = bootstrapping_200_2.sort_values(by = "predicción del modelo", ascending = False)
    bootstrapping_200_2 = bootstrapping_200_2.head(200)

    suma_2 = bootstrapping_200_2["product"].sum()
    multiplicacion_2 = suma_2 * 4500
    resta_2 = multiplicacion_2 - 100000000

    ganancias_boot_region_2.append(resta_2)
# Verificación de la longitud de la muestra Bootstrapping
len(ganancias_boot_region_2)

# Cálculo de las ganancias promedio de la región
ganancias_promedio_boot_region_2 = np.mean(ganancias_boot_region_2)

print("Las ganancias promedio de la región 2 con el boostrapping son de:", ganancias_promedio_boot_region_2 )

# Determinación del intervalo de confianza
inferior = np.percentile(ganancias_boot_region_2, 2.5)
superior = np.percentile(ganancias_boot_region_2, 97.5)

print("IC 95%:")
print(inferior, superior)


# Cálculo del porcentaje del riesgo de pérdidas
riesgo_perdidas_2 = (np.array(ganancias_boot_region_2) < 0).mean() * 100

print("El riesgo de pérdidas para la región 2 es del", riesgo_perdidas_2, "%")

# Cálculo de porcentaje de la probabilidad de ganancia
probabilidad_ganancia_2 = (np.array(ganancias_boot_region_2) > 0).mean() * 100

print("La probabilidad de ganancias para la región 2 es del", probabilidad_ganancia_2, "%")

<class 'pandas.core.frame.DataFrame'>
Index: 25000 entries, 71751 to 43558
Data columns (total 2 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   predicción del modelo  25000 non-null  float64
 1   product                25000 non-null  float64
dtypes: float64(2)
memory usage: 1.1 MB
None
       predicción del modelo     product
71751              93.599633   61.212375
80493              75.105159   41.850118
2655               90.066809   57.776581
53233             105.162375  100.053761
91141             115.303310  109.897122
Las ganancias promedio de la región 2 con el boostrapping son de: 4044038.665683568
IC 95%:
-1633504.1339559986 9503595.749237997
El riesgo de pérdidas para la región 2 es del 7.6 %
La probabilidad de ganancias para la región 2 es del 92.4 %


Para la región 2 encontramos que las ganancias promedio se calculan alrededor de 4.04 millones de dólares, aunque los resultados del bootstrapping nos señalan que con un índice de confianza del 95% en 1000 simulaciones la empresa se mantuvo en escenarios donde obtuvo pérdidas de 1.63 millones con una probabilidad del 7.6% de dólares u obtuvo ganancias estimadas en 9.50 millones de dólares con una probabilidad del 92.4%.

El objetivo de este proyecto fue identificar la región más adecuada para el desarrollo de 200 nuevos pozos petrolíferos mediante modelos de regresión lineal y un análisis de riesgo basado en la técnica de bootstrapping.

En la etapa de modelado, la región 1 presentó el mejor desempeño predictivo, obteniendo un RECM considerablemente menor que las regiones 0 y 2. Esto indicó que las características disponibles explicaban con mucha mayor precisión el volumen de reservas existentes en dicha región.

Posteriormente, se calcularon las ganancias potenciales utilizando los 200 pozos con las mayores reservas estimadas. En esta fase, la región 0 mostró la mayor ganancia potencial. Sin embargo, este resultado representaba únicamente un escenario favorable y no consideraba la incertidumbre inherente a la selección de pozos.

Por esta razón, se aplicó la técnica de bootstrapping con 1000 simulaciones para estimar de forma más realista el beneficio esperado y el riesgo asociado a cada región.

Los resultados obtenidos fueron los siguientes:

Región 0

Ganancia promedio: aproximadamente 3.97 millones de dólares.
Riesgo de pérdidas: 6.9%.
No cumple el criterio de riesgo establecido por el negocio.

Región 1

Ganancia promedio: aproximadamente 4.56 millones de dólares.
Intervalo de confianza del 95% completamente positivo.
Riesgo de pérdidas: 1.5%.
Cumple el criterio de riesgo establecido por el negocio.

Región 2

Ganancia promedio: aproximadamente 4.04 millones de dólares.
Riesgo de pérdidas: 7.6%.
No cumple el criterio de riesgo establecido por el negocio.

De acuerdo con las condiciones del negocio, solo deben considerarse regiones cuyo riesgo de pérdidas sea inferior al 2.5%. Bajo este criterio, las regiones 0 y 2 quedan descartadas debido a que presentan riesgos significativamente superiores al límite permitido.

Por lo tanto, la región 1 es la región recomendada para el desarrollo de nuevos pozos petrolíferos, ya que es la única que cumple con el requisito de riesgo del negocio y, al mismo tiempo, presenta la mayor ganancia promedio entre las regiones que satisfacen dicho criterio.

En conclusión, aunque la región 0 mostró inicialmente la mayor ganancia potencial, el análisis de incertidumbre realizado mediante bootstrapping demostró que la región 1 ofrece la mejor combinación de rentabilidad y seguridad, convirtiéndose en la alternativa más adecuada para la inversión de la empresa.